# ResNet50 — tea-leaf disease classification

Seven-class image-level classification using pretrained TorchVision ResNet50. This notebook preserves the training/evaluation approach in the original Kaggle export. Set the dataset path before running. 

In [ ]:
# ============================================================
# Tea Leaf Disease Classification (Kaggle End-to-End Pipeline)
# TorchVision ResNet50 pretrained (NO HuggingFace needed)
# Train / Val / Test Split + Early stopping + Confusion Matrix
# ============================================================

import os, random, time
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

## Reproducibility

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1) Dataset path (auto-detect)

In [ ]:
BASE_CANDIDATES = [
    "/kaggle/input/tealeafbd-tea-leaf-disease-detection/teaLeafBD/teaLeafBD",
    "/kaggle/input/tealeafbd-tea-leaf-disease-detection/teaLeafBD",
]

data_root = None
for p in BASE_CANDIDATES:
    if os.path.isdir(p):
        subdirs = [d for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))]
        if len(subdirs) >= 2:
            data_root = p
            break

if data_root is None:
    raise FileNotFoundError("Could not find dataset folder. Update BASE_CANDIDATES to your dataset path.")

print("Data root:", data_root)

## 2) Build dataframe

In [ ]:
class_names = sorted([d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))])
num_classes = len(class_names)
print("Classes:", class_names)

label2idx = {c: i for i, c in enumerate(class_names)}
idx2label = {i: c for c, i in label2idx.items()}

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
rows = []
for c in class_names:
    cdir = os.path.join(data_root, c)
    for fn in os.listdir(cdir):
        if fn.lower().endswith(IMG_EXT):
            rows.append({"path": os.path.join(cdir, fn), "label": label2idx[c], "class": c})

df = pd.DataFrame(rows)
print("Total images:", len(df))
print("\nClass distribution:\n", df["class"].value_counts())

## 3) Stratified Train/Val/Test split (70/15/15)

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])
print(f"\nSplit sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

## 4) Dataset

In [ ]:
class ImageDFDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        label = int(self.df.loc[idx, "label"])
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, path

## 5) Transforms (ImageNet normalization)

In [ ]:
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

train_ds = ImageDFDataset(train_df, transform=train_tfms)
val_ds   = ImageDFDataset(val_df, transform=val_tfms)
test_ds  = ImageDFDataset(test_df, transform=val_tfms)

## 6) DataLoaders

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## 7) Model: TorchVision ResNet50 pretrained

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights)

# Replace final classifier
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(device)

print("Using TorchVision pretrained: ResNet50 (IMAGENET1K_V2)")

## 8) Loss + Optimizer + Scheduler + AMP

In [ ]:
# Class weights to handle imbalance
class_counts = train_df["label"].value_counts().sort_index().values
class_weights = (1.0 / class_counts)
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

EPOCHS = 20
steps_per_epoch = len(train_loader)
total_steps = max(1, steps_per_epoch * EPOCHS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

## 9) Train/Eval loops

In [ ]:
def run_one_epoch(model, loader, train=True):
    model.train() if train else model.eval()

    all_preds, all_labels = [], []
    running_loss = 0.0

    for images, labels, _paths in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    avg_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_labels, all_preds

# Early stopping + best checkpoint
best_val_acc = -1.0
best_path = "best_tealeaf_resnet50.pth"
patience = 5
pat_counter = 0

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc, _, _ = run_one_epoch(model, train_loader, train=True)
    va_loss, va_acc, _, _ = run_one_epoch(model, val_loader, train=False)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    dt = time.time() - t0
    print(f"Epoch [{epoch:02d}/{EPOCHS}]  "
          f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}  "
          f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}  "
          f"time={dt:.1f}s")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({
            "arch": "resnet50",
            "state_dict": model.state_dict(),
            "class_names": class_names,
            "img_size": IMG_SIZE
        }, best_path)
        pat_counter = 0
        print("  -> Saved best model.")
    else:
        pat_counter += 1
        if pat_counter >= patience:
            print(f"Early stopping triggered. Best val_acc={best_val_acc:.4f}")
            break

print("\nBest model saved at:", best_path, "best_val_acc:", best_val_acc)

## 10) Plot training curves

In [ ]:
plt.figure()
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure()
plt.plot(history["train_acc"], label="train_acc")
plt.plot(history["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 11) Load best model

In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval()
print("Loaded best checkpoint.")

## 12) Evaluation

In [ ]:
def evaluate_and_report(model, loader, split_name="VAL"):
    loss, acc, y_true, y_pred = run_one_epoch(model, loader, train=False)

    print(f"\n[{split_name}] loss={loss:.4f} acc={acc:.4f}\n")
    target_names = [idx2label[i] for i in range(num_classes)]
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Confusion Matrix ({split_name})")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(num_classes), target_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), target_names)
    plt.colorbar()
    plt.tight_layout()
    plt.show()

    return y_true, y_pred

_ = evaluate_and_report(model, val_loader, "VAL")
_ = evaluate_and_report(model, test_loader, "TEST")

## 13) Show random test predictions

In [ ]:
def denorm(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = img_tensor.cpu() * std + mean
    return torch.clamp(x, 0, 1)

sample_idx = np.random.choice(len(test_ds), size=min(8, len(test_ds)), replace=False)

plt.figure(figsize=(14, 7))
for i, idx in enumerate(sample_idx, start=1):
    img, label, _path = test_ds[idx]
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
        pred = torch.argmax(logits, dim=1).item()

    plt.subplot(2, 4, i)
    plt.imshow(denorm(img).permute(1, 2, 0))
    plt.axis("off")
    plt.title(f"GT: {idx2label[label]}\nPR: {idx2label[pred]}", fontsize=10)

plt.tight_layout()
plt.show()

print("\nDone. Best model file:", best_path)